# xlingual-drift: RQ3 seam runner

**Question (RQ3):** at which stage, and at which language crossing ("seam"), does the extra
non-English failure happen? The same 20 long-horizon instances (shop + pay) run in zh, hi, ta and
Hinglish under three seams (project doc Section 8.5):

| Seam | Request | Tool outputs | Error messages |
|---|---|---|---|
| S1 | task language | English | English |
| S2 | task language | task language | English |
| S3 | task language | task language | task language |

English episodes from the RQ1 run serve as S0. Comparisons are **paired by instance** (same task,
same language, different seam) with an exact McNemar test, and every failed constraint is assigned a
stage from the failure taxonomy (Section 9): entity handling, state carryover (and its semantic
form), recovery / re-grounding, or reasoning / comprehension.

Run all on an A100. 240 episodes; `--resume` continues across sessions.

In [ ]:
# 1. Which GPU did Colab give us? The run config decides whether it is big enough (cell 5).
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2. Clone or update the repo. The repo is private, so this uses a GitHub token
#    stored in Colab Secrets as GITHUB_TOKEN (key icon in the left sidebar).
import os
from google.colab import userdata
token = userdata.get("GITHUB_TOKEN")
if not os.path.exists("/content/xlingual-drift"):
    !git clone -q https://{token}@github.com/khalerishabh/xlingual-drift.git /content/xlingual-drift
else:
    !cd /content/xlingual-drift && git pull -q
%cd /content/xlingual-drift
!git log --oneline -1

In [ ]:
# 3. Install. The harness needs only light packages, in Colab's own Python.
#    vLLM goes into a separate venv so it never touches Colab's preinstalled
#    torch/torchaudio/RAPIDS; --torch-backend=auto picks the PyTorch build that
#    matches this machine's GPU driver. The vLLM version is pinned by the run config,
#    because it is part of what defines a backbone.
!pip install -q -r requirements.txt
!pip install -q uv
import yaml
CONFIG = "configs/rq3_seams_qwen36_27b_fp8.yaml"
VLLM_VERSION = yaml.safe_load(open(CONFIG))["serving"]["vllm_version"]
if not os.path.exists("/content/vllm-env/bin/vllm"):
    !uv venv -q --python 3.12 /content/vllm-env
    !uv pip install -q -p /content/vllm-env/bin/python "vllm=={VLLM_VERSION}" --torch-backend=auto
# ninja: vLLM compiles some kernels at warm-up (e.g. the FP8 fallback on A100) and needs it.
!uv pip install -q -p /content/vllm-env/bin/python ninja
!/content/vllm-env/bin/python -c "import torch, vllm; print('torch', torch.__version__, '| CUDA build', torch.version.cuda, '| GPU visible', torch.cuda.is_available(), '| vllm', vllm.__version__)"

In [ ]:
# 4. Raw logs go to Drive so a disconnect never loses finished episodes.
from google.colab import drive
drive.mount("/content/drive")
RUNS_DIR = "/content/drive/MyDrive/xlingual-drift-runs"
os.makedirs(RUNS_DIR, exist_ok=True)

## Start the model server

The model, weights revision, parsers and minimum GPU memory come from the run config's
`serving` block, so the notebook can only serve what the config pins. Gate v2 uses the
official **Qwen3.6-27B-FP8**, which fits a 40GB or 80GB A100. On a smaller GPU the cell stops
and asks you to switch runtime. The first start downloads the weights (roughly 30GB).
Server output goes to `/content/vllm.log`.

In [ ]:
# 5. Serve exactly the backbone the run config pins, and record what was served.
#    A backbone is the model plus its exact weights and precision; run_grid
#    refuses to mix episodes from different setups in one log.
import json, subprocess, time, urllib.request, yaml
from huggingface_hub import HfApi

serving = yaml.safe_load(open(CONFIG))["serving"]

gpu_name, vram_mib = [x.strip() for x in subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True).stdout.split(",")]
vram_gb = int(vram_mib) / 1024
if vram_gb < serving["min_vram_gb"]:
    raise SystemExit(f"{serving['model']} needs >= {serving['min_vram_gb']} GB of GPU memory; this runtime has "
                     f"{gpu_name} ({vram_gb:.0f} GB). Disconnect, choose A100, and reconnect.")

sha = HfApi().model_info(serving["model"], revision=serving["revision"]).sha
vllm_version = subprocess.run(["/content/vllm-env/bin/python", "-c", "import vllm; print(vllm.__version__)"],
                              capture_output=True, text=True).stdout.strip()
if vllm_version != serving["vllm_version"]:
    raise SystemExit(f"vLLM {vllm_version} installed but the config pins {serving['vllm_version']}. "
                     "Delete /content/vllm-env and rerun cell 3.")
SERVING_JSON = "/content/serving.json"
json.dump({"model": serving["model"], "revision": sha, "vllm_version": vllm_version,
           "gpu_name": gpu_name, "vram_gb": round(vram_gb), "max_model_len": serving["max_model_len"]},
          open(SERVING_JSON, "w"), indent=2)
print(open(SERVING_JSON).read())

cmd = ["/content/vllm-env/bin/vllm", "serve", serving["model"],
       "--revision", sha,
       "--port", "8000",
       "--max-model-len", str(serving["max_model_len"]),
       "--gpu-memory-utilization", "0.90",
       "--enable-auto-tool-choice",
       "--tool-call-parser", serving["tool_call_parser"],
       "--reasoning-parser", serving["reasoning_parser"],
       *serving.get("extra_args", [])]
import os
# The venv's bin must be on PATH so vLLM finds build tools (ninja) it installed there.
env = {**os.environ, "PATH": "/content/vllm-env/bin:" + os.environ["PATH"]}
server = subprocess.Popen(cmd, stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT, env=env)

for i in range(240):
    try:
        if urllib.request.urlopen("http://localhost:8000/health", timeout=5).status == 200:
            print(f"server up after ~{i * 10}s")
            break
    except Exception:
        pass
    if server.poll() is not None:
        raise RuntimeError("vLLM exited; see the tail of /content/vllm.log below")
    time.sleep(10)
else:
    raise TimeoutError("server not up after 40 min")

In [ ]:
# If the server failed to start, the reason is here.
!tail -n 30 /content/vllm.log

## Checks before spending GPU time

In [ ]:
# 6. Harness sanity: tests and the long-horizon mock smoke test (no model involved).
!python -m pytest -q
!python -m eval.run_grid --config configs/smoke_test_long.yaml --out /content/smoke_test_long.jsonl --workers 8 | tail -2

## Run the three seams

In [ ]:
# 7. Run. --serving stamps every episode; --resume continues a partial log.
import json
RUN = yaml.safe_load(open(CONFIG))
LOG = f"{RUNS_DIR}/{RUN['run_name']}.jsonl"
print("config:", CONFIG, "| log:", LOG)
!python -m eval.run_grid --config {CONFIG} --serving {SERVING_JSON} --resume --workers {16 if not RUN['horizons'] else 32} --out {LOG}

In [ ]:
# 8. RQ3 analysis: success by seam (paired), and the stage of every failed constraint per seam.
#    English S0 reference: the RQ1 log, if it is in Drive.
import os
from eval.drift import load, report, seam_report
RESCORED = LOG.replace(".jsonl", ".rescored.jsonl")
!python -m eval.rescore_long {LOG} {RESCORED}
records = load(RESCORED)
RQ1_LOG = f"{RUNS_DIR}/rq1_long_qwen36_27b_fp8.jsonl"
english = [r for r in load(RQ1_LOG) if r["language"] == "en"] if os.path.exists(RQ1_LOG) else []
print(seam_report(records, english))
print()
print(report(records))

## Reading the result

- **S1 vs S2** isolates the tool-output crossing: S2 removes it. If S2 succeeds more often than S1 for
  a language, English tool outputs cost that language accuracy.
- **S2 vs S3** isolates the error-message crossing (the pay domain always produces one IMPS error,
  and any validation errors the agent triggers).
- The stage table says *where* the failures sit under each seam. The project doc hypothesis (RQ3) is
  that excess failure concentrates in state carryover and entity handling, not comprehension, and
  that the tool-output seam costs more than the request seam.

In [ ]:
# 9. Download the raw log.
from google.colab import files
files.download(LOG)